# Is the few-shot gain learning, or priming by the classes shown?

The sweep found that three in-prompt examples raise the judge's agreement with the first
annotator from kappa = 0.115 to 0.238. The three examples were one S, one F3 and one F7,
and the resulting judge assigns S to 66 trajectories, F7 to 49 and F3 to 34, where the
human assigns F7 to 18. Three of the four codes it uses most are the three it was shown.
That pattern has two readings: the examples taught it the rubric, or they shifted its
label prior toward whatever appeared in the prompt.

This notebook separates them. Everything is held fixed, including the model, the
decoding, the ground-truth block, the scoring code and the number of examples; only the
classes of the three examples change, to F2, F5 and F1, which are disjoint from the first
set. The examples still come from the 30-trajectory pilot and never from the 200 being
scored.

The prediction under learning is that kappa stays near 0.238 and the assignment profile
stays close to the human one. The prediction under priming is that kappa moves and the
inflated classes follow the new examples.

**Requires:** accelerator GPU T4 x2, internet on.


In [ ]:
import ast, gc, json, time, urllib.request
import torch
from collections import Counter

BASE = ("https://raw.githubusercontent.com/achenna-source/"
        "icv-agent-observability/neucom-d-26-17269-r1/")
fetch = lambda p: urllib.request.urlopen(BASE + p, timeout=90).read().decode("utf-8")

ns = {}
exec(compile(fetch("code/judge.py"), "judge.py", "exec"), ns)
build_prompt, parse_reply, GEN = ns["build_prompt"], ns["parse_reply"], ns["GENERATION_KWARGS"]
mns = {}
exec(compile(fetch("code/metrics.py"), "metrics.py", "exec"), mns)
kappa_lenient, cluster_bootstrap = mns["kappa_lenient"], mns["cluster_bootstrap"]

rows = [json.loads(l) for l in fetch("data/trajectories_200.jsonl").splitlines() if l.strip()]
pilot = [json.loads(l) for l in fetch("data/trajectories_pilot_30.jsonl").splitlines() if l.strip()]
_kb = json.loads(fetch("data/knowledge_base.json"))
kb, cities_absent = _kb["facts"], _kb["populations_absent_by_design"]
steps_of = lambda r: ast.literal_eval(r["trajectory"]) if isinstance(r["trajectory"], str) else r["trajectory"]

human = [r["label_human"] for r in rows]
valid = set(human)
clusters = [r.get("cluster_id", r["id"]) for r in rows]
print("evaluation corpus %d | pilot %d | facts %d" % (len(rows), len(pilot), len(kb)))
print("human label counts: %s" % dict(Counter(human).most_common()))
print("gpus: %d" % torch.cuda.device_count())


## The two example sets

Set A is the one the sweep used. Set B is this control. Both take the first pilot
trajectory carrying each wanted code, in file order, so no example is chosen on how well
the judge is expected to do with it.


In [ ]:
SET_A = ["S", "F3", "F7"]
SET_B = ["F2", "F5", "F1"]

def pick(codes):
    out = []
    for c in codes:
        for r in pilot:
            if r["label_human"] == c:
                out.append(r)
                break
    assert len(out) == len(codes), "a wanted code is absent from the pilot"
    return out

def shot_block(shots):
    out = []
    for r in shots:
        p = build_prompt({"task": r["task"], "trajectory": steps_of(r)}, kb, cities_absent)
        body = p.split("TRAJECTOIRE :", 1)[1].split("Reponds UNIQUEMENT", 1)[0].strip()
        out.append('EXEMPLE :\nTRAJECTOIRE :\n%s\nReponse : {"code":"%s"}'
                   % (body, r["label_human"]))
    return "\n\n".join(out)

BLOCKS = {}
for name, codes in (("A", SET_A), ("B", SET_B)):
    shots = pick(codes)
    BLOCKS[name] = shot_block(shots)
    print("set %s: %s | block %d characters"
          % (name, [(r["id"], r["label_human"]) for r in shots], len(BLOCKS[name])))

def make_prompt(r, block):
    p = build_prompt({"task": r["task"], "trajectory": steps_of(r)}, kb, cities_absent)
    if block is None:
        return p
    head, tail = p.split("TRAJECTOIRE :", 1)
    return head + block + "\n\nTRAJECTOIRE :" + tail


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-7B-Instruct"
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16,
                                             device_map="auto")
model.eval()
print("%s loaded in %.0f s" % (MODEL, time.time() - t0))

def run(block, tag):
    preds = []
    t0 = time.time()
    for i, r in enumerate(rows):
        msgs = [{"role": "user", "content": make_prompt(r, block)}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        enc = tok(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, pad_token_id=tok.eos_token_id, **GEN)
        preds.append(parse_reply(tok.decode(out[0][enc["input_ids"].shape[1]:],
                                            skip_special_tokens=True)))
        if (i + 1) % 50 == 0:
            print("   [%s] %3d/200  %.0f s" % (tag, i + 1, time.time() - t0))
    return [p if p else "UNPARSED" for p in preds]

def score(pred, tag):
    agree = sum(1 for a, b in zip(human, pred) if a == b)
    k = kappa_lenient(human, pred, valid)
    items = [(c, (h, p)) for c, h, p in zip(clusters, human, pred)]
    lo, hi, _ = cluster_bootstrap(items, lambda d: kappa_lenient(
        [h for h, _ in d], [p for _, p in d], valid))
    print("[%s] agreement %d/200 = %.1f%% | kappa %.3f | 95%% CI [%.3f, %.3f]"
          % (tag, agree, 100.0 * agree / 200, k, lo, hi))
    print("      codes assigned: %s" % dict(Counter(pred).most_common(8)))
    return {"tag": tag, "agreement": agree, "kappa": round(k, 3),
            "ci": [round(lo, 3), round(hi, 3)], "preds": pred}

RESULTS = []


## The control run

Set B only. Set A was already run on this same model in the sweep, so the half hour of
GPU buys a new configuration instead of repeating one. Its per-code counts are restated
below from that run, not recomputed here.


In [ ]:
p = run(BLOCKS["B"], "7B few-shot set B")
RESULTS.append(score(p, "7B few-shot set B"))
json.dump(RESULTS, open("judge_priming.json", "w"))


## Does the inflation follow the examples?

For each configuration, the count the judge assigns to each code is compared with the
count the first annotator assigns. If the examples prime, the codes shown should be the
ones most over-assigned, and that over-assignment should move when the shown codes move.


In [ ]:
# Set A counts as measured in the sweep run, same model, same prompt builder, same
# decoding; restated so the two profiles can be read side by side.
A_COUNTS = {"S": 66, "F7": 49, "F1": 36, "F3": 34, "F5": 10, "F2": 2, "F4": 2, "F4,F3": 1}

hum = Counter(human)
B_COUNTS = dict(Counter(RESULTS[-1]["preds"]))
codes = sorted(set(list(hum) + ["S", "F1", "F2", "F3", "F4", "F5", "F6", "F7"]))

print("%-5s %7s %12s %12s" % ("code", "human", "few-shot A", "few-shot B"))
for c in codes:
    print("%-5s %7d %12d %12d"
          % (c, hum.get(c, 0), A_COUNTS.get(c, 0), B_COUNTS.get(c, 0)))
print()
print("assigned against human count, for the codes shown in each set")
for name, shown, cnt in (("A", SET_A, A_COUNTS), ("B", SET_B, B_COUNTS)):
    for c in shown:
        h = hum.get(c, 0)
        print("   set %s, shown %-3s : assigned %3d, human %3d, ratio %s"
              % (name, c, cnt.get(c, 0), h,
                 ("%.2f" % (cnt.get(c, 0) / h)) if h else "n/a"))
print()
print("assigned against human count, for the codes NOT shown in set B")
for c in codes:
    if c in SET_B:
        continue
    h = hum.get(c, 0)
    print("   %-3s : assigned %3d, human %3d, ratio %s"
          % (c, B_COUNTS.get(c, 0), h,
             ("%.2f" % (B_COUNTS.get(c, 0) / h)) if h else "n/a"))


In [ ]:
print("=== for the manuscript ===")
print("%-22s %10s %8s %18s" % ("configuration", "agreement", "kappa", "95% interval"))
print("%-22s %9.1f%% %8.3f %18s" % ("7B zero-shot", 30.0, 0.115, "published"))
print("%-22s %9.1f%% %8.3f %18s" % ("3B zero-shot", 28.5, 0.100, "[0.012, 0.196]"))
print("%-22s %9.1f%% %8.3f %18s" % ("Falcon3 7B zero-shot", 28.5, 0.038, "[-0.037, 0.118]"))
print("%-22s %9.1f%% %8.3f %18s" % ("7B few-shot set A", 39.0, 0.238, "[0.122, 0.357]"))
r = RESULTS[-1]
print("%-22s %9.1f%% %8.3f   [%+.3f, %+.3f]"
      % ("7B few-shot set B", 100.0 * r["agreement"] / 200, r["kappa"],
         r["ci"][0], r["ci"][1]))
